# Day113: ORQA inference on Colab

Runs entirely inside this Colab VM -- the ORQA repo, the Qwen2-VL-2B-Instruct base model, and the smallest distilled checkpoint are all downloaded here, never touching local storage. Uses LLaMA-Factory's `ChatModel` API (the same code path the ORQA authors' own inference pipeline is built on) instead of hand-rolled loading code, since it already knows how to combine the quantized base model with the LoRA adapter and flash-attention correctly.

**Before running**: Runtime -> Change runtime type -> T4 GPU (or better). This will not work on a CPU runtime (bitsandbytes 4-bit + flash-attn both need CUDA).

In [ ]:
!nvidia-smi

## 1. Clone ORQA and install dependencies

Skips the point-cloud stack (`spconv`, `torch-scatter`, etc.) entirely -- those are CUDA-build-toolchain-specific and we only need the image+text path for this test.

In [ ]:
import os
# Day113 fix: pip's -q flag was swallowing wandb's interactive login prompt text
# during the LLaMA-Factory install, leaving a blank input box with no visible
# question. We don't use wandb for this inference-only test, so disable it
# up front instead of answering the (invisible) prompt.
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

# Day113 fix: always work from a fixed absolute path. Re-running this cell
# without a full runtime restart previously nested clones inside each other
# (/content/ORQA/Qwen2-VL/LLaMA-Factory/ORQA/Qwen2-VL/LLaMA-Factory/...)
# because %cd ORQA is relative to wherever the cell happened to already be.
os.chdir("/content")
if not os.path.isdir("/content/ORQA"):
    !git clone --depth 1 https://github.com/egeozsoy/ORQA.git
else:
    print("/content/ORQA already exists, skipping clone")

# Day113 fix, take 3: one single pip call (so the resolver sees every
# constraint at once) still left numpy in a state where something in this set
# wants numpy>=2 (numpy.dtypes.StringDType, added in 2.0) while something else
# was built against numpy<2's ABI ("dtype size changed, ... 96 ... 88").
# Pinning numpy explicitly to an early 2.x release (2.0.2) is the last thing
# worth trying: it has StringDType (satisfies the >=2 side) while still being
# close enough to the 1.x ABI transition that packages merely "built without
# accounting for 2.0 yet" often still work against it in practice, unlike
# newer 2.x releases that have moved further away.
os.chdir("/content/ORQA/Qwen2-VL/LLaMA-Factory")
!pip install -q "numpy==2.0.2" qwen-vl-utils==0.0.2 transformers==4.46.1 bitsandbytes==0.44.1 accelerate -e ".[torch,metrics]"
print("cwd:", os.getcwd())

## 2. Download the smallest distilled checkpoint (Dist-S, ~278M distilled params)

Straight from Hugging Face into the Colab VM's own disk -- nothing routed through local storage.

In [ ]:
import os
os.chdir("/content/ORQA/Qwen2-VL/LLaMA-Factory")  # Day113 fix: always pin absolute path
!mkdir -p saves
!wget -q --show-progress -O saves/dist_s.zip "https://huggingface.co/egeozsoy/ORQA/resolve/main/checkpoints/qwen2vl_lora_sft_qlora_1000000_unfreeze8_0.5mmdrop_336res_578imgtoks_pkd_050_depthreduce4.zip?download=true"
!unzip -q saves/dist_s.zip -d saves/
!ls saves/

## 3. Get a test image

ORQA was trained on room-level operating-room views (ceiling/wall cameras seeing the whole OR: people, tools, robot arms) -- not laparoscopic/endoscopic in-body footage. Using one frame from this project's own stomach-phantom or CMR Surgical cholecystectomy data is a genuine out-of-domain test, not the task ORQA was built for; a room-level OR photo would be the fairer comparison. Trying both is worthwhile precisely because the gap tells us something.

Cell below grabs one frame from a public OR-style demo image (swap the URL for your own room-level photo if you have one) plus, for contrast, one frame from the stomach-phantom endoscope data this playground series already used.

In [ ]:
import os
os.chdir("/content/ORQA/Qwen2-VL/LLaMA-Factory")  # Day113 fix: always pin absolute path
import cv2
import urllib.request

# in-domain-ish test: one frame from this project's Day106 stomach-phantom endoscope data
url = (
    "https://huggingface.co/datasets/nvidia/PhysicalAI-Robotics-Open-H-Embodiment/resolve/main/"
    "Endoscopy/cuhk/openh_dataset_full/find_greater_curvature/videos/chunk-000/"
    "observation.images.endo%E4%B8%89/episode_000000.mp4"
)
urllib.request.urlretrieve(url, "test_episode.mp4")
cap = cv2.VideoCapture("test_episode.mp4")
cap.set(cv2.CAP_PROP_POS_FRAMES, 50)
ok, frame = cap.read()
cv2.imwrite("test_frame.jpg", frame)
print("saved test_frame.jpg", ok)

## 4. Load ORQA (base + LoRA adapter) and ask it about the image

Adjust `adapter_name_or_path` below to whatever directory `unzip` produced in step 2 if it differs -- check the `saves/` listing printed above.

In [ ]:
import os
os.chdir("/content/ORQA/Qwen2-VL/LLaMA-Factory")  # Day113 fix: always pin absolute path
import sys
# Day113 fix: belt-and-suspenders for "ModuleNotFoundError: No module named
# 'llamafactory'" -- if the earlier editable install's registration didn't
# survive a runtime restart or a re-run from a different cwd, this makes sure
# the source directory is importable regardless.
if "/content/ORQA/Qwen2-VL/LLaMA-Factory/src" not in sys.path:
    sys.path.insert(0, "/content/ORQA/Qwen2-VL/LLaMA-Factory/src")

import glob
from llamafactory.chat import ChatModel

adapter_dir = glob.glob("saves/*depthreduce4*/checkpoint-*") or glob.glob("saves/*depthreduce4*")
print("using adapter:", adapter_dir)

args = dict(
    model_name_or_path="Qwen/Qwen2-VL-2B-Instruct",
    adapter_name_or_path=adapter_dir[0],
    template="qwen2_vl",
    infer_backend="huggingface",
    quantization_bit=4,
    flash_attn="sdpa",  # Day113: flash-attn wouldn't build on Colab; falling back to SDPA
)
chat_model = ChatModel(args)

In [ ]:
messages = [{"role": "user", "content": "Describe what you see in this image. What is happening, and what tools or structures are visible?"}]
response = chat_model.chat(messages, images=["test_frame.jpg"])
print(response[0].response_text)

## Notes for interpreting the output

- If the response is generic/wrong (hallucinated OR layout, people who aren't there, etc.), that's expected and informative -- it would confirm ORQA is as domain-specific as the paper's own leave-one-out results suggest, not a generalist model that happens to also handle endoscopic views.
- We're running on SDPA instead of flash-attn (see Day113 notes above) -- the training config's own comment warned SDPA "simply does not work" for this model, so if the output is degraded/incoherent/repetitive, this attention-backend mismatch (not a code bug) is the most likely reason, not the domain gap discussed above.
- Runtime -> Disconnect and delete runtime when done, so nothing about this session persists anywhere -- consistent with keeping this entirely off local storage in the first place.